# 16 - Jina CLIP v2 モデル評価

> **⚠️ スキップ**: このノートブックは評価対象外です。Jina CLIP のモデルコードが現在の `transformers` ライブラリのバージョンと互換性がないため（`create_position_ids_from_input_ids` のインポートエラー）、評価をスキップしました。

## 概要
Jina AI の CLIP v2 多言語モデルを評価する。

## モデル情報
- **Model**: jinaai/jina-clip-v2
- **Embedding dimension**: 1024
- **Type**: Multimodal (Image + Text, 89言語対応)
- **特徴**: 日本語を含む89言語対応、512x512入力対応

In [2]:
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

from image_vector_poc import JinaCLIPEmbedder
from image_vector_poc.evaluation import EvaluationReporter, evaluate_embeddings

## 設定

In [3]:
DB_PATH = Path("../data/images.duckdb")
OUTPUT_DIR = Path("../data/evaluations")
BATCH_SIZE = 16  # Larger model, smaller batch size
RANDOM_STATE = 42

## 画像カタログの読み込み

In [4]:
conn = duckdb.connect(str(DB_PATH), read_only=True)
query = """
    SELECT id, file_path, category, file_name
    FROM image_catalog
    ORDER BY category, file_name
"""
catalog = conn.execute(query).fetchall()
conn.close()

image_ids = [r[0] for r in catalog]
file_paths = [r[1] for r in catalog]
categories = [r[2] for r in catalog]

category_labels = np.array(categories)
category_counts = pd.Series(categories).value_counts().to_dict()
unique_categories = list(category_counts.keys())

print(f"Total images: {len(catalog)}")
print(f"Categories: {unique_categories}")

Total images: 385
Categories: ['EuroPython2025', 'PyConJP2025', 'PyConJP2025-PreCampHiroshima', 'KashiwaVillagePark2026', 'TokyoNight202505', 'terada']


## モデルの初期化

In [5]:
print("Loading Jina CLIP v2 model...")
embedder = JinaCLIPEmbedder(device="cuda")
print(f"Model: {embedder.model_name}")
print(f"Embedding dimension: {embedder.embedding_dim}")
print(f"Device: {embedder.device}")

Loading Jina CLIP v2 model...


config.json: 0.00B [00:00, ?B/s]

configuration_clip.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-clip-implementation:
- configuration_clip.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


modeling_clip.py: 0.00B [00:00, ?B/s]

eva_model.py: 0.00B [00:00, ?B/s]

rope_embeddings.py: 0.00B [00:00, ?B/s]

Encountered exception while importing einops: No module named 'einops'


ImportError: This modeling file requires the following packages that were not found in your environment: einops. Run `pip install einops`

## 画像のベクトル化

In [ ]:
print(f"Generating embeddings for {len(file_paths)} images...")

start_time = time.time()
embeddings_list = []

for i in tqdm(range(0, len(file_paths), BATCH_SIZE), desc="Embedding"):
    batch_paths = file_paths[i:i + BATCH_SIZE]
    batch_images = []
    for path in batch_paths:
        try:
            img = Image.open(path).convert("RGB")
            batch_images.append(img)
        except Exception as e:
            print(f"Error loading {path}: {e}")
            batch_images.append(Image.new("RGB", (224, 224), color="gray"))
    
    batch_embs = embedder.embed_images(batch_images)
    embeddings_list.append(batch_embs)

embeddings = np.vstack(embeddings_list)
processing_time = time.time() - start_time

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Processing time: {processing_time:.2f}s")
print(f"Speed: {len(file_paths) / processing_time:.1f} images/sec")

## 評価の実行

In [ ]:
print("Running evaluation...\n")

metrics = evaluate_embeddings(
    embeddings=embeddings,
    labels=category_labels,
    model_name=embedder.model_name,
    embedding_dim=embedder.embedding_dim,
    categories=unique_categories,
    category_counts=category_counts,
    processing_time=processing_time,
    random_state=RANDOM_STATE,
)

## 結果の表示

In [ ]:
print("=" * 60)
print("Evaluation Results - Jina CLIP v2")
print("=" * 60)
print(f"Model: {metrics.model_name}")
print(f"Embedding dimension: {metrics.embedding_dim}")
print(f"Processing time: {metrics.processing_time_seconds:.2f}s")

print("\n--- t-SNE Metrics ---")
print(f"2D: Silhouette={metrics.silhouette_2d:.4f}, Trust={metrics.trustworthiness_2d:.4f}, DistRatio={metrics.distance_ratio_2d:.4f}")
print(f"3D: Silhouette={metrics.silhouette_3d:.4f}, Trust={metrics.trustworthiness_3d:.4f}, DistRatio={metrics.distance_ratio_3d:.4f}")

print("\n--- PCA Metrics ---")
print(f"2D: Silhouette={metrics.pca_silhouette_2d:.4f}, Variance={metrics.pca_variance_ratio_2d:.4f}")
print(f"3D: Silhouette={metrics.pca_silhouette_3d:.4f}, Variance={metrics.pca_variance_ratio_3d:.4f}")

## 多言語テキスト埋め込みのテスト

In [ ]:
# 多言語テキストのテスト
test_queries = [
    ("English", "a photo of a cat"),
    ("Japanese", "猫の写真"),
    ("German", "ein Foto einer Katze"),
    ("French", "une photo d'un chat"),
]

print("Multilingual text embedding test:")
for lang, query in test_queries:
    text_emb = embedder.embed_text(query)
    print(f"  [{lang}] '{query}': shape={text_emb.shape}, norm={np.linalg.norm(text_emb):.4f}")

## 結果の保存

In [ ]:
reporter = EvaluationReporter(OUTPUT_DIR)
filepath = reporter.save(metrics)
print(f"Results saved to: {filepath}")

## GPUメモリのクリーンアップ

In [ ]:
del embedder
del embeddings

import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared.")